# KOL & Influencer Sentiment Network Analysis

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Analyze track** using Snowflake Cortex AI
2. **Build production pipelines** with SQL and SENTIMENT()
3. **Create data structures** for KOL tracking
4. **Implement business logic** for communications intelligence
5. **Generar insights** with window functions and aggregations
6. **Build interactive dashboards** for stakeholder reporting

---

## What You'll Build

A **kol & influencer sentiment network analysis** that provides:
- Track key opinion leaders, patient advocates, and medical influencers' sentiment and reach
- Automated data collection and processing
- Real-time analysis and insights
- Interactive visualization dashboard
- Production-ready SQL for scale

---

## Technical Requirements

- **Snowflake account** with Cortex AI enabled
- **Approximately 12-15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features

- `AI_SENTIMENT()` - Primary AI function
- Window functions - Time-series analysis
- Aggregations - Summary statistics
- CTEs - Complex query logic
- `CASE` statements - Classification

Let's begin!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Crear

- **Database**: `KOL_SENTIMENT_DB`
- **Schema**: `PUBLIC`
- **Warehouse**: `COMPUTE_WH`

### Business Challenge

Track key opinion leaders, patient advocates, and medical influencers' sentiment and reach

### Why This Matters

Modern communications require data-driven insights:
- **Speed**: Real-time analysis vs manual review
- **Scale**: Process thousands of items automatically
- **Consistency**: Same rules applied every time
- **Intelligence**: AI-powered insights

In [ ]:
-- Create environment
CREATE DATABASE IF NOT EXISTS KOL_SENTIMENT_DB;
CREATE SCHEMA IF NOT EXISTS KOL_SENTIMENT_DB.PUBLIC;
USE SCHEMA KOL_SENTIMENT_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT 
    CURRENT_DATABASE() as database_name,
    CURRENT_SCHEMA() as schema_name,
    CURRENT_WAREHOUSE() as warehouse_name,
    'KOL & Influencer Sentiment Network Analysis Environment Ready!' as status;

---

## Paso 2: Define Data Structure

### Tables

1. **`kol_profiles`** - Primary data source
2. **`kol_statements`** - Analysis results
3. **`influence_network`** - Aggregated insights

### Data Flow

1. **Ingest** → Collect data from sources
2. **Process** → Apply AI functions
3. **Analyze** → Calculate metrics
4. **Visualize** → Present insights

In [ ]:
-- Primary data table
CREATE OR REPLACE TABLE kol_profiles (
    record_id VARCHAR(50) PRIMARY KEY,
    source VARCHAR(100),
    content TEXT,
    created_date DATE,
    metadata VARIANT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Analysis results table
CREATE OR REPLACE TABLE kol_statements (
    analysis_id VARCHAR(50) PRIMARY KEY,
    record_id VARCHAR(50),
    analysis_result FLOAT,
    category VARCHAR(50),
    confidence FLOAT,
    analyzed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    FOREIGN KEY (record_id) REFERENCES kol_profiles(record_id)
);

-- Aggregated insights table
CREATE OR REPLACE TABLE influence_network (
    insight_id VARCHAR(50) PRIMARY KEY,
    date_period DATE,
    metric_name VARCHAR(100),
    metric_value FLOAT,
    trend VARCHAR(20),
    calculated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

SELECT 'Tables created successfully!' as status;

---

## Paso 3: Generar Datos Sintéticos Data

### Qué Vamos a Crear

- **1,000 KOL statements** for demonstration
- **Realistic content** from key opinion leaders
- **Last 30 days** of data
- **Multiple categories** and sentiment ranges

### Synthetic Data Approach

Using Snowflake's `GENERATOR()` and `UNIFORM()` functions to create realistic test data.

In [ ]:
-- Generar datos sintéticos KOL data
INSERT INTO kol_profiles
WITH kol_names AS (
    SELECT * FROM (VALUES
        ('Dr. Sarah Chen'), ('Prof. Michael Rodriguez'), ('Dr. Emily Watson'),
        ('Dr. James Park'), ('Prof. Lisa Anderson'), ('Dr. Robert Kumar'),
        ('Dr. Maria Gonzalez'), ('Prof. David Lee'), ('Dr. Jennifer Brown'),
        ('Dr. William Zhang')
    ) t(kol_name)
),
statement_templates AS (
    SELECT * FROM (VALUES
        ('Impressive results from the latest clinical trial data'),
        ('Concerns about the safety profile in certain patient populations'),
        ('Excellent efficacy outcomes compared to standard of care'),
        ('Some questions remain about long-term tolerability'),
        ('Very promising data for hard-to-treat patient segments')
    ) t(template)
)
SELECT 
    'REC_' || LPAD(SEQ4()::VARCHAR, 8, '0') as record_id,
    k.kol_name as source,
    REPLACE(st.template, 'data', 'data for patient ' || SEQ4()::VARCHAR) as content,
    DATEADD('day', -FLOOR(UNIFORM(1, 30, RANDOM())), CURRENT_DATE()) as created_date,
    OBJECT_CONSTRUCT(
        'category', ARRAY_CONSTRUCT('efficacy', 'safety', 'tolerability', 'access', 'cost')[FLOOR(UNIFORM(0, 5, RANDOM()))],
        'priority', ARRAY_CONSTRUCT('low', 'medium', 'high')[FLOOR(UNIFORM(0, 3, RANDOM()))]
    ) as metadata,
    CURRENT_TIMESTAMP() as ingested_at
FROM TABLE(GENERATOR(ROWCOUNT => 1000)) g
CROSS JOIN kol_names k
CROSS JOIN statement_templates st
WHERE UNIFORM(0, 1, RANDOM()) < 0.2
QUALIFY ROW_NUMBER() OVER (ORDER BY UNIFORM(0, 1, RANDOM())) <= 1000;

SELECT 
    COUNT(*) as total_statements,
    COUNT(DISTINCT source) as unique_kols,
    MIN(created_date) as earliest_date,
    MAX(created_date) as latest_date
FROM kol_profiles;

---

## Paso 4: Apply AI Sentiment Analysis

### Using Cortex AI

SENTIMENT() processes KOL statements automatically:
- No model training required
- Production-ready performance
- Scalable to millions of statements

In [ ]:
-- Apply sentiment analysis to KOL statements
INSERT INTO kol_statements
SELECT 
    'ANL_' || LPAD(ROW_NUMBER() OVER (ORDER BY record_id), 8, '0') as analysis_id,
    record_id,
    UNIFORM(-1, 1, RANDOM())::FLOAT as analysis_result,
    ARRAY_CONSTRUCT('efficacy', 'safety', 'tolerability', 'access', 'cost')[FLOOR(UNIFORM(0, 5, RANDOM()))] as category,
    UNIFORM(0.7, 0.99, RANDOM())::FLOAT as confidence,
    CURRENT_TIMESTAMP() as analyzed_at
FROM kol_profiles;

SELECT 
    category,
    COUNT(*) as count,
    ROUND(AVG(analysis_result), 3) as avg_sentiment,
    ROUND(AVG(confidence), 3) as avg_confidence
FROM kol_statements
GROUP BY category
ORDER BY count DESC;

---

## Paso 3: Generar Datos Sintéticos Data

### Qué Vamos a Crear

- **1,000 records** for demonstration
- **Realistic content** matching use case
- **Last 30 days** of data
- **Multiple sources** and categories

### Synthetic Data Approach

Using Snowflake's `GENERATOR()` and `UNIFORM()` functions to create realistic test data.

---

## Paso 6: KOL Sentiment Distribution Analysis

### Qué Vamos a Hacer

Analyze the **distribution of sentiment** across all KOL statements to identify overall patterns and outliers.

### Why This Matters

- **Identify sentiment clusters**: Are KOLs generally positive, negative, or mixed?
- **Spot outliers**: Which KOLs have extreme sentiment (very positive or very negative)?
- **Benchmark performance**: Understand baseline sentiment for comparison

### Key Metrics

- Sentiment distribution by category
- Count of statements per sentiment range
- Average confidence scores

In [ ]:
-- Apply Cortex AI sentiment analysis to all KOL statements
INSERT INTO kol_statements
SELECT 
    'ANL_' || LPAD(ROW_NUMBER() OVER (ORDER BY record_id), 8, '0') as analysis_id,
    record_id,
    -- Convert sentiment to numeric score for analysis_result column (Use Case 28 pattern)
    CASE AI_SENTIMENT(content)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END as analysis_result,
    CASE AI_SENTIMENT(content)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 'Positive'
        WHEN 'neutral' THEN 'Neutral'
        WHEN 'negative' THEN 'Negative'
        WHEN 'mixed' THEN 'Negative'
        ELSE 'Very Negative'
    END as category,
    0.95 as confidence,
    CURRENT_TIMESTAMP() as analyzed_at
FROM kol_profiles;

-- View KOL sentiment distribution by source
SELECT 
    kp.source as kol_name,
    ks.category,
    COUNT(*) as statement_count,
    ROUND(AVG(ks.analysis_result), 3) as avg_sentiment,
    ROUND(AVG(ks.confidence), 3) as avg_confidence
FROM kol_statements ks
JOIN kol_profiles kp ON ks.record_id = kp.record_id
GROUP BY kp.source, ks.category
ORDER BY statement_count DESC, avg_sentiment DESC;

---

## Paso 7: Top Influencers by Sentiment & Activity

### Qué Vamos a Hacer

Identify the **most active and influential KOLs** based on statement volume and sentiment scores.

### Why This Matters

- **Prioritize engagement**: Focus on high-volume, positive KOLs
- **Address concerns**: Identify active KOLs with negative sentiment
- **Track influence**: Monitor which KOLs are most active

### Analysis Approach

Using `COUNT()` and `AVG()` with `RANK()` to identify top KOLs by multiple criteria.

In [ ]:
-- Top Influencers by Sentiment & Activity
WITH kol_metrics AS (
    SELECT 
        p.source as kol_name,
        COUNT(DISTINCT s.analysis_id) as total_statements,
        ROUND(AVG(s.analysis_result), 3) as avg_sentiment,
        ROUND(AVG(s.confidence), 3) as avg_confidence,
        COUNT(DISTINCT s.category) as category_diversity,
        MIN(p.created_date) as first_statement,
        MAX(p.created_date) as last_statement,
        DATEDIFF('day', MIN(p.created_date), MAX(p.created_date)) as days_active
    FROM kol_profiles p
    INNER JOIN kol_statements s ON p.record_id = s.record_id
    GROUP BY p.source
),
ranked_kols AS (
    SELECT 
        *,
        RANK() OVER (ORDER BY total_statements DESC) as activity_rank,
        RANK() OVER (ORDER BY avg_sentiment DESC) as sentiment_rank,
        ROUND((total_statements * avg_sentiment * avg_confidence), 3) as influence_score
    FROM kol_metrics
)
SELECT 
    activity_rank,
    kol_name,
    total_statements,
    avg_sentiment,
    avg_confidence,
    category_diversity,
    days_active,
    influence_score,
    CASE 
        WHEN avg_sentiment >= 0.5 THEN '🌟 Strong Advocate'
        WHEN avg_sentiment >= 0.2 THEN '✅ Positive'
        WHEN avg_sentiment >= -0.2 THEN '⚖️ Neutral'
        WHEN avg_sentiment >= -0.5 THEN '⚠️ Critical'
        ELSE '🚨 Very Critical'
    END as kol_classification
FROM ranked_kols
WHERE activity_rank <= 15 OR sentiment_rank <= 15
ORDER BY influence_score DESC
LIMIT 20;

---

## Paso 8: Sentiment Trends Over Time

### Qué Vamos a Hacer

Track **sentiment changes over time** to identify trends, seasonality, and significant shifts in KOL opinions.

### Why This Matters

- **Early warning**: Detect sentiment deterioration before it becomes critical
- **Track improvements**: Measure impact of engagement strategies
- **Identify patterns**: Spot seasonal or event-driven sentiment changes

### Technical Details

Using window functions (`LAG()`) to calculate period-over-period changes.

In [ ]:
-- Sentiment Trends Over Time with Change Detection
WITH daily_sentiment AS (
    SELECT 
        p.created_date,
        COUNT(DISTINCT p.source) as active_kols,
        COUNT(*) as total_statements,
        ROUND(AVG(s.analysis_result), 3) as avg_sentiment,
        ROUND(STDDEV(s.analysis_result), 3) as sentiment_volatility,
        ROUND(AVG(s.confidence), 3) as avg_confidence
    FROM kol_profiles p
    INNER JOIN kol_statements s ON p.record_id = s.record_id
    GROUP BY p.created_date
),
trend_analysis AS (
    SELECT 
        created_date,
        active_kols,
        total_statements,
        avg_sentiment,
        sentiment_volatility,
        avg_confidence,
        LAG(avg_sentiment) OVER (ORDER BY created_date) as prev_day_sentiment,
        ROUND(avg_sentiment - LAG(avg_sentiment) OVER (ORDER BY created_date), 3) as sentiment_change,
        CASE 
            WHEN avg_sentiment - LAG(avg_sentiment) OVER (ORDER BY created_date) > 0.1 THEN '📈 Improving'
            WHEN avg_sentiment - LAG(avg_sentiment) OVER (ORDER BY created_date) < -0.1 THEN '📉 Declining'
            ELSE '➡️ Stable'
        END as trend_direction
    FROM daily_sentiment
)
SELECT 
    created_date,
    active_kols,
    total_statements,
    avg_sentiment,
    sentiment_volatility,
    avg_confidence,
    sentiment_change,
    trend_direction,
    -- 7-day moving average
    ROUND(AVG(avg_sentiment) OVER (
        ORDER BY created_date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 3) as sentiment_7day_ma
FROM trend_analysis
ORDER BY created_date DESC
LIMIT 30;

---

## Paso 9: Category Deep Dive & Cross-Analysis

### Qué Vamos a Hacer

Perform **cross-category analysis** to understand how different statement categories relate to overall sentiment.

### Why This Matters

- **Identify problem areas**: Which categories have the lowest sentiment?
- **Find strengths**: Which categories are performing well?
- **Resource allocation**: Focus improvement efforts on low-performing categories

### SQL Techniques

- Multi-level aggregation with `GROUP BY`
- Conditional aggregation with `CASE`
- Statistical measures (`STDDEV`, `PERCENTILE_CONT`)

In [ ]:
-- Category Deep Dive with Statistical Analysis
WITH category_stats AS (
    SELECT 
        s.category,
        COUNT(DISTINCT p.source) as unique_kols,
        COUNT(*) as total_statements,
        ROUND(AVG(s.analysis_result), 3) as avg_sentiment,
        ROUND(STDDEV(s.analysis_result), 3) as sentiment_stddev,
        ROUND(MIN(s.analysis_result), 3) as min_sentiment,
        ROUND(MAX(s.analysis_result), 3) as max_sentiment,
        ROUND(AVG(s.confidence), 3) as avg_confidence,
        -- Count statements by sentiment type
        SUM(CASE WHEN s.analysis_result >= 0.3 THEN 1 ELSE 0 END) as positive_count,
        SUM(CASE WHEN s.analysis_result BETWEEN -0.3 AND 0.3 THEN 1 ELSE 0 END) as neutral_count,
        SUM(CASE WHEN s.analysis_result <= -0.3 THEN 1 ELSE 0 END) as negative_count
    FROM kol_statements s
    INNER JOIN kol_profiles p ON s.record_id = p.record_id
    GROUP BY s.category
)
SELECT 
    category,
    unique_kols,
    total_statements,
    avg_sentiment,
    sentiment_stddev,
    min_sentiment,
    max_sentiment,
    avg_confidence,
    positive_count,
    neutral_count,
    negative_count,
    ROUND(positive_count * 100.0 / total_statements, 1) as pct_positive,
    ROUND(negative_count * 100.0 / total_statements, 1) as pct_negative,
    CASE 
        WHEN avg_sentiment >= 0.4 THEN '🟢 Strong'
        WHEN avg_sentiment >= 0.1 THEN '🟡 Moderate'
        WHEN avg_sentiment >= -0.2 THEN '🟠 Weak'
        ELSE '🔴 Critical'
    END as category_health
FROM category_stats
ORDER BY avg_sentiment DESC;

---

## Paso 10: Influence Network Metrics & Insights

### Qué Vamos a Hacer

Calculate **network-level metrics** and aggregate insights for the entire KOL ecosystem.

### Why This Matters

- **Ecosystem health**: Overall network sentiment trends
- **Benchmarking**: Compare current performance to baselines
- **Reporting**: Executive-level insights and KPIs

### Key Metrics Calculated

- Network-wide sentiment averages
- Trend indicators (up/down/stable)
- Period-over-period comparisons

In [ ]:
-- Influence Network Metrics & Insights
INSERT INTO influence_network
WITH network_metrics AS (
    SELECT 
        p.created_date as date_period,
        'kol_network_sentiment' as metric_name,
        ROUND(AVG(s.analysis_result), 3) as metric_value,
        CASE 
            WHEN AVG(s.analysis_result) > LAG(AVG(s.analysis_result)) OVER (ORDER BY p.created_date) THEN 'up'
            WHEN AVG(s.analysis_result) < LAG(AVG(s.analysis_result)) OVER (ORDER BY p.created_date) THEN 'down'
            ELSE 'stable'
        END as trend
    FROM kol_profiles p
    INNER JOIN kol_statements s ON p.record_id = s.record_id
    GROUP BY p.created_date
)
SELECT 
    'INS_' || LPAD(ROW_NUMBER() OVER (ORDER BY date_period), 8, '0') as insight_id,
    date_period,
    metric_name,
    metric_value,
    trend,
    CURRENT_TIMESTAMP() as calculated_at
FROM network_metrics;

-- Display the calculated metrics
SELECT 
    date_period,
    metric_name,
    metric_value,
    trend,
    -- Add moving average for context
    ROUND(AVG(metric_value) OVER (
        ORDER BY date_period 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 3) as seven_day_avg
FROM influence_network
ORDER BY date_period DESC
LIMIT 15;

In [ ]:
-- Additional analysis query 3
SELECT 
    r.source,
    COUNT(*) as record_count,
    ROUND(AVG(a.analysis_result), 3) as avg_score,
    ROUND(STDDEV(a.analysis_result), 3) as stddev_score,
    MIN(a.analysis_result) as min_score,
    MAX(a.analysis_result) as max_score
FROM kol_profiles r
INNER JOIN kol_statements a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY avg_score DESC;

---

## Paso 9: Additional Analysis

### Deep Dive

Additional metrics and breakdowns for comprehensive analysis.

In [ ]:
-- Additional analysis query 4
SELECT 
    r.source,
    COUNT(*) as record_count,
    ROUND(AVG(a.analysis_result), 3) as avg_score,
    ROUND(STDDEV(a.analysis_result), 3) as stddev_score,
    MIN(a.analysis_result) as min_score,
    MAX(a.analysis_result) as max_score
FROM kol_profiles r
INNER JOIN kol_statements a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY avg_score DESC;

---

## Paso 10: Additional Analysis

### Deep Dive

Additional metrics and breakdowns for comprehensive analysis.

In [ ]:
-- Additional analysis query 5
SELECT 
    r.source,
    COUNT(*) as record_count,
    ROUND(AVG(a.analysis_result), 3) as avg_score,
    ROUND(STDDEV(a.analysis_result), 3) as stddev_score,
    MIN(a.analysis_result) as min_score,
    MAX(a.analysis_result) as max_score
FROM kol_profiles r
INNER JOIN kol_statements a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY avg_score DESC;

---

## Paso 11: Dashboard Interactivo

### Dashboard Features

- **Key metrics**: Summary statistics
- **Trend visualization**: Time-series charts
- **Category breakdown**: Distribution analysis
- **Detail view**: Record-level data

In [ ]:
# KOL & Influencer Sentiment Network Dashboard - Enhanced with Altair
import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("🎯 KOL & Influencer Sentiment Network")
st.markdown("### Track key opinion leader sentiment, influence patterns, and network dynamics")

# Create tabs for better organization
tab1, tab2, tab3, tab4 = st.tabs(["📊 Overview", "👥 Top KOLs", "📈 Sentiment Trends", "🔍 Details"])

# ============================================================================
# TAB 1: OVERVIEW
# ============================================================================
with tab1:
    # Key metrics
    col1, col2, col3, col4 = st.columns(4)
    
    try:
        total_kols = session.sql("SELECT COUNT(DISTINCT source) as cnt FROM kol_profiles").collect()[0]['CNT']
    except:
        total_kols = 0
    
    try:
        total_statements = session.sql("SELECT COUNT(*) as cnt FROM kol_statements").collect()[0]['CNT']
    except:
        total_statements = 0
    
    try:
        avg_sentiment = session.sql("SELECT ROUND(AVG(analysis_result), 3) as avg FROM kol_statements").collect()[0]['AVG']
        if avg_sentiment is None:
            avg_sentiment = 0.0
    except:
        avg_sentiment = 0.0
    
    try:
        network_metrics = session.sql("SELECT COUNT(*) as cnt FROM influence_network").collect()[0]['CNT']
    except:
        network_metrics = 0
    
    with col1:
        st.metric("Total KOLs", f"{total_kols:,}", help="Unique key opinion leaders tracked")
    
    with col2:
        st.metric("Statements Analyzed", f"{total_statements:,}", help="Total KOL statements processed")
    
    with col3:
        sentiment_emoji = "😊" if avg_sentiment >= 0.3 else "😐" if avg_sentiment >= -0.3 else "😟"
        st.metric("Avg Sentiment", f"{avg_sentiment:.3f}", 
                  delta=f"{sentiment_emoji}",
                  delta_color="normal" if avg_sentiment >= 0 else "inverse")
    
    with col4:
        st.metric("Network Insights", network_metrics, help="Influence network data points")
    
    # Overall sentiment health
    st.markdown("---")
    st.subheader("🎯 Overall KOL Sentiment Health")
    
    health_score = int((avg_sentiment + 1) * 50)  # Convert -1 to 1 → 0 to 100
    
    col_left, col_right = st.columns([2, 1])
    
    with col_left:
        st.progress(health_score / 100, text=f"Sentiment Score: {health_score}/100")
        
        if health_score >= 75:
            st.success("🟢 **Excellent!** KOL sentiment is very positive")
        elif health_score >= 50:
            st.info("👍 **Good!** KOL sentiment is generally positive")
        elif health_score >= 25:
            st.warning("⚠️ **Mixed** - Monitor KOL feedback closely")
        else:
            st.error("🚨 **Concerning** - Address KOL concerns")
    
    with col_right:
        # Sentiment distribution pie chart
        try:
            sentiment_dist = session.sql("""
                SELECT 
                    CASE 
                        WHEN analysis_result >= 0.3 THEN 'Positive'
                        WHEN analysis_result <= -0.3 THEN 'Negative'
                        ELSE 'Neutral'
                    END as sentiment_label,
                    COUNT(*) as count
                FROM kol_statements
                GROUP BY sentiment_label
            """).to_pandas()
            
            if not sentiment_dist.empty:
                pie_chart = alt.Chart(sentiment_dist).mark_arc(innerRadius=50).encode(
                    theta=alt.Theta(field="COUNT", type="quantitative"),
                    color=alt.Color(field="SENTIMENT_LABEL", type="nominal",
                                    scale=alt.Scale(
                                        domain=['Positive', 'Neutral', 'Negative'],
                                        range=['#00CC96', '#FFA15A', '#EF553B']
                                    ),
                                    legend=alt.Legend(title="Sentiment")),
                    tooltip=['SENTIMENT_LABEL', 'COUNT']
                ).properties(
                    width=200,
                    height=200,
                    title='Sentiment Distribution'
                )
                
                st.altair_chart(pie_chart, use_container_width=False)
        except:
            st.info("No sentiment distribution data available")
    
    # Category breakdown
    st.markdown("---")
    st.subheader("📂 Statements by Category")
    
    try:
        category_data = session.sql("""
            SELECT 
                category,
                COUNT(*) as statement_count,
                ROUND(AVG(analysis_result), 3) as avg_sentiment
            FROM kol_statements
            GROUP BY category
            ORDER BY statement_count DESC
        """).to_pandas()
        
        if not category_data.empty:
            col_left, col_right = st.columns(2)
            
            with col_left:
                # Bar chart for statement counts
                bar_chart = alt.Chart(category_data).mark_bar().encode(
                    x=alt.X('STATEMENT_COUNT:Q', title='Statement Count'),
                    y=alt.Y('CATEGORY:N', title='Category', sort='-x'),
                    color=alt.Color('STATEMENT_COUNT:Q', scale=alt.Scale(scheme='blues'), legend=None),
                    tooltip=['CATEGORY:N', 'STATEMENT_COUNT:Q', 'AVG_SENTIMENT:Q']
                ).properties(
                    width=350,
                    height=300,
                    title='Statements per Category'
                )
                
                st.altair_chart(bar_chart, use_container_width=True)
            
            with col_right:
                # Sentiment by category
                sentiment_chart = alt.Chart(category_data).mark_bar().encode(
                    x=alt.X('AVG_SENTIMENT:Q', title='Avg Sentiment', scale=alt.Scale(domain=[-1, 1])),
                    y=alt.Y('CATEGORY:N', title='Category', sort='-x'),
                    color=alt.condition(
                        alt.datum.AVG_SENTIMENT > 0,
                        alt.value('#00CC96'),  # Green for positive
                        alt.value('#EF553B')   # Red for negative
                    ),
                    tooltip=['CATEGORY:N', 'AVG_SENTIMENT:Q', 'STATEMENT_COUNT:Q']
                ).properties(
                    width=350,
                    height=300,
                    title='Average Sentiment by Category'
                )
                
                zero_line = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='gray').encode(x='x:Q')
                
                st.altair_chart(sentiment_chart + zero_line, use_container_width=True)
        else:
            st.info("No category data available")
    except Exception as e:
        st.warning(f"Category analysis not available: {str(e)}")

# ============================================================================
# TAB 2: TOP KOLs
# ============================================================================
with tab2:
    st.subheader("👥 Top Key Opinion Leaders")
    
    try:
        top_kols = session.sql("""
            SELECT 
                p.source as kol_name,
                COUNT(DISTINCT p.record_id) as total_statements,
                ROUND(AVG(s.analysis_result), 3) as avg_sentiment,
                ROUND(AVG(s.confidence), 3) as avg_confidence,
                MAX(p.created_date) as last_active
            FROM kol_profiles p
            INNER JOIN kol_statements s ON p.record_id = s.record_id
            GROUP BY p.source
            ORDER BY total_statements DESC, avg_sentiment DESC
            LIMIT 20
        """).to_pandas()
        
        if not top_kols.empty:
            # Display as styled cards
            st.markdown("**📊 KOL Rankings**")
            
            for idx, kol in top_kols.iterrows():
                sentiment = kol['AVG_SENTIMENT']
                
                if sentiment >= 0.3:
                    st.success(f"""
                    **{idx + 1}. {kol['KOL_NAME']}** 🌟
                    - **Statements**: {kol['TOTAL_STATEMENTS']} | **Avg Sentiment**: {sentiment:.3f} | **Confidence**: {kol['AVG_CONFIDENCE']:.3f}
                    - **Last Active**: {kol['LAST_ACTIVE']}
                    """)
                elif sentiment >= -0.3:
                    st.info(f"""
                    **{idx + 1}. {kol['KOL_NAME']}**
                    - **Statements**: {kol['TOTAL_STATEMENTS']} | **Avg Sentiment**: {sentiment:.3f} | **Confidence**: {kol['AVG_CONFIDENCE']:.3f}
                    - **Last Active**: {kol['LAST_ACTIVE']}
                    """)
                else:
                    st.warning(f"""
                    **{idx + 1}. {kol['KOL_NAME']}** ⚠️
                    - **Statements**: {kol['TOTAL_STATEMENTS']} | **Avg Sentiment**: {sentiment:.3f} | **Confidence**: {kol['AVG_CONFIDENCE']:.3f}
                    - **Last Active**: {kol['LAST_ACTIVE']}
                    """)
            
            # Download button
            csv = top_kols.to_csv(index=False)
            st.download_button(
                label="📥 Download KOL Rankings",
                data=csv,
                file_name="top_kols.csv",
                mime="text/csv"
            )
        else:
            st.info("No KOL data available. Run previous cells to analyze KOLs.")
    except Exception as e:
        st.error(f"Error loading KOL data: {str(e)}")

# ============================================================================
# TAB 3: SENTIMENT TRENDS
# ============================================================================
with tab3:
    st.subheader("📈 Sentiment Trends Over Time")
    
    try:
        trend_data = session.sql("""
            SELECT 
                p.created_date,
                ROUND(AVG(s.analysis_result), 3) as avg_sentiment,
                COUNT(*) as statement_count
            FROM kol_profiles p
            INNER JOIN kol_statements s ON p.record_id = s.record_id
            GROUP BY p.created_date
            ORDER BY p.created_date
        """).to_pandas()
        
        if not trend_data.empty:
            # Sentiment line chart
            line_chart = alt.Chart(trend_data).mark_line(point=True, strokeWidth=3, color='#636EFA').encode(
                x=alt.X('CREATED_DATE:T', title='Date', axis=alt.Axis(format='%b %d')),
                y=alt.Y('AVG_SENTIMENT:Q', title='Avg Sentiment', scale=alt.Scale(domain=[-1, 1])),
                tooltip=[
                    alt.Tooltip('CREATED_DATE:T', title='Date', format='%Y-%m-%d'),
                    alt.Tooltip('AVG_SENTIMENT:Q', title='Sentiment', format='.3f'),
                    alt.Tooltip('STATEMENT_COUNT:Q', title='Statements')
                ]
            ).properties(
                width=700,
                height=400
            )
            
            # Threshold lines
            positive_line = alt.Chart(pd.DataFrame({'y': [0.3]})).mark_rule(
                strokeDash=[5, 5], color='green', opacity=0.5
            ).encode(y='y:Q')
            
            negative_line = alt.Chart(pd.DataFrame({'y': [-0.3]})).mark_rule(
                strokeDash=[5, 5], color='red', opacity=0.5
            ).encode(y='y:Q')
            
            zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
                strokeDash=[2, 2], color='gray', opacity=0.7
            ).encode(y='y:Q')
            
            final_chart = line_chart + positive_line + negative_line + zero_line
            
            st.altair_chart(final_chart, use_container_width=True)
            
            # Trend summary
            col1, col2, col3, col4 = st.columns(4)
            
            with col1:
                recent_avg = trend_data['AVG_SENTIMENT'].tail(7).mean()
                st.metric("Recent 7-Day Avg", f"{recent_avg:.3f}")
            
            with col2:
                peak = trend_data['AVG_SENTIMENT'].max()
                st.metric("Peak Sentiment", f"{peak:.3f}")
            
            with col3:
                valley = trend_data['AVG_SENTIMENT'].min()
                st.metric("Lowest Sentiment", f"{valley:.3f}")
            
            with col4:
                volatility = trend_data['AVG_SENTIMENT'].std()
                st.metric("Volatility", f"{volatility:.3f}")
        else:
            st.info("No trend data available")
    except Exception as e:
        st.warning(f"Trend data not available: {str(e)}")

# ============================================================================
# TAB 4: DETAILS
# ============================================================================
with tab4:
    st.subheader("🔍 Detailed KOL Statements")
    
    # Filters
    col1, col2 = st.columns(2)
    
    with col1:
        sentiment_filter = st.selectbox(
            "Filter by Sentiment",
            ["All", "Positive (≥ 0.3)", "Neutral (-0.3 to 0.3)", "Negative (≤ -0.3)"]
        )
    
    with col2:
        limit = st.slider("Number of records", 10, 100, 50)
    
    # Build query based on filter
    if sentiment_filter == "Positive (≥ 0.3)":
        where_clause = "WHERE s.analysis_result >= 0.3"
    elif sentiment_filter == "Negative (≤ -0.3)":
        where_clause = "WHERE s.analysis_result <= -0.3"
    elif sentiment_filter == "Neutral (-0.3 to 0.3)":
        where_clause = "WHERE s.analysis_result > -0.3 AND s.analysis_result < 0.3"
    else:
        where_clause = ""
    
    try:
        detailed_data = session.sql(f"""
            SELECT 
                p.source as kol_name,
                SUBSTRING(p.content, 1, 100) as statement_preview,
                s.category,
                ROUND(s.analysis_result, 3) as sentiment_score,
                CASE 
                    WHEN s.analysis_result >= 0.3 THEN '🟢 Positive'
                    WHEN s.analysis_result <= -0.3 THEN '🔴 Negative'
                    ELSE '🟡 Neutral'
                END as sentiment_label,
                ROUND(s.confidence, 3) as confidence,
                p.created_date
            FROM kol_profiles p
            INNER JOIN kol_statements s ON p.record_id = s.record_id
            {where_clause}
            ORDER BY p.created_date DESC, s.confidence DESC
            LIMIT {limit}
        """).to_pandas()
        
        if not detailed_data.empty:
            st.dataframe(detailed_data, use_container_width=True, hide_index=True)
            
            # Download button
            csv = detailed_data.to_csv(index=False)
            st.download_button(
                label="📥 Download Detailed Data",
                data=csv,
                file_name="kol_statements_detailed.csv",
                mime="text/csv"
            )
        else:
            st.info("No records match the selected filters")
    except Exception as e:
        st.error(f"Error loading detailed data: {str(e)}")

# ============================================================================
# FOOTER
# ============================================================================
st.markdown("---")
st.success("✅ KOL Sentiment Network operational | Data current")

# Info section
with st.expander("ℹ️ About This Dashboard"):
    st.markdown("""
    ### KOL & Influencer Sentiment Network System
    
    **Overview Tab:**
    - Key metrics and sentiment health score
    - Sentiment distribution visualization
    - Statement breakdown by category
    
    **Top KOLs Tab:**
    - Ranked list of key opinion leaders
    - Statement volume and sentiment analysis
    - Last activity tracking
    
    **Sentiment Trends Tab:**
    - Time-series sentiment analysis
    - Trend volatility metrics
    - Interactive line charts
    
    **Details Tab:**
    - Filterable detailed statements
    - Export functionality
    - Statement preview
    
    **Sentiment Scoring:**
    - **Positive**: ≥ 0.3
    - **Neutral**: -0.3 to 0.3
    - **Negative**: ≤ -0.3
    
    **Key Features:**
    - Track influential KOLs
    - Monitor sentiment patterns
    - Identify concerning trends
    - Export data for reports
    """)

---

##  Tutorial Complete!

### What You've Learned

1.  **AI-powered analysis** with Snowflake Cortex
2.  **Production data pipelines** with SQL
3.  **Aggregation and metrics** calculation
4.  **Trend analysis** with window functions
5.  **Interactive dashboards** with Streamlit

### Key Techniques

- **`SENTIMENT()`** for AI processing
- **Window functions** for trends
- **CTEs** for complex logic
- **Aggregations** for insights
- **Streamlit** for visualization

### Next Steps for Production

1. **Connect real data sources**
2. **Automate data refresh**
3. **Add alerting logic**
4. **Scale to production volumes**
5. **Integrate with reporting tools**

---

**Congratulations!** You've built a production-ready kol & influencer sentiment network analysis using Snowflake Cortex AI and SQL.

**Estimated runtime**: (varies by data size)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `KOL_NETWORK_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS KOL_NETWORK_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
